In [ ]:
# connecting the cluster 
import ipyparallel as parallel
clients = parallel.Client() 
print("Connected engine IDs:", clients.ids)

Connected engine IDs: [0, 1, 2]


In [ ]:
# executing the code on all engines
%%px
from mpi4py import MPI
import numpy as np

comm = MPI.COMM_WORLD
rank = comm.Get_rank()
size = comm.Get_size() # This will be 3

# STEP 1: Loading the inputs into process 0

data_array = None
kernel_array = None
data_shape = None

if rank == 0:
    # Creating a 6x6 input data array and a 3x3 kernel for demonstration
    data_array = np.ones((6, 6)) 
    kernel_array = np.array([[1, 0, -1], 
                             [1, 0, -1], 
                             [1, 0, -1]])
    data_shape = data_array.shape


# STEPS 2 & 3: Broadcast dimensions and kernel

# Send dimensions and kernel from process 0 to all other processes
data_shape = comm.bcast(data_shape, root=0)
kernel_array = comm.bcast(kernel_array, root=0)

# STEPS 4 & 5: Split and Scatter data
local_data = None
if rank == 0:
    # Split rows equally: a 6x6 array split across 3 processes gives 2 rows per process
    split_data = np.array_split(data_array, size, axis=0) 
else:
    split_data = None

# Scatter the specific chunks to corresponding processes
local_data = comm.scatter(split_data, root=0)

